# EP14. Agent-to-Agent 통신: MCP + A2A

## 학습 목표

1. **MCP와 A2A 프로토콜**의 역할 차이를 이해하고 적절히 구분한다
2. **Google A2A 프로토콜** 구조(Agent Card, Task, Artifact)를 분석하고 구현한다
3. **A2A HTTP 서버/클라이언트**를 구축하여 에이전트 간 통신을 구현한다
4. **MCP + A2A 결합 아키텍처**로 도구 연결과 에이전트 협업을 통합한다

## AI 가이드

> 막히는 부분은 Claude에게 물어보세요.
>
> - "A2A Agent Card에 새로운 스킬을 추가하는 방법을 알려줘"
> - "MCP Tool 안에서 A2A 에이전트를 호출하는 패턴을 설명해줘"

**사전 요구사항**: EP12 MCP 기본 이해, Python 비동기 프로그래밍 기초

**예상 소요 시간**: 75분

**필요한 API 키**:
- `ANTHROPIC_API_KEY`
- `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY`

---
## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install fastmcp httpx langchain-anthropic langfuse pydantic python-dotenv --quiet

---
## 섹션 2. 라이브러리 로드 + API 키 설정

In [ ]:
import os
import json
import uuid
import asyncio
from datetime import datetime
from enum import Enum
from typing import Optional

import httpx
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# .env 파일에서 API 키 로드
load_dotenv()

assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY가 설정되지 않았습니다"
assert os.getenv("LANGFUSE_PUBLIC_KEY"), "LANGFUSE_PUBLIC_KEY가 설정되지 않았습니다"

print("API 키 로드 완료 ✓")

---
## 섹션 3. MCP vs A2A 개념 비교

MCP와 A2A는 **상호 보완적** 프로토콜입니다.

| 구분 | MCP | A2A |
|------|-----|-----|
| **목적** | LLM ↔ 도구 연결 | 에이전트 ↔ 에이전트 통신 |
| **비유** | USB-C 케이블 | 전화번호 교환 |
| **단위** | Tool, Resource, Prompt | Task, Artifact, Message |
| **통신** | JSON-RPC (stdio/HTTP) | HTTP + JSON |
| **발표** | Anthropic (2024) | Google (2024) |
| **상태 관리** | Stateless | Task 상태 추적 |

In [ ]:
# MCP vs A2A 비교: 개념 코드로 차이점 이해

# === MCP 패턴: LLM이 도구를 호출 ===
mcp_example = {
    "protocol": "MCP",
    "direction": "LLM → Tool (단방향 호출)",
    "example": {
        "method": "tools/call",
        "params": {
            "name": "query_db",
            "arguments": {"sql": "SELECT * FROM products LIMIT 5"}
        }
    },
    "use_case": "DB 조회, 파일 읽기, API 호출 등 도구 접근"
}

# === A2A 패턴: 에이전트가 다른 에이전트에게 작업 요청 ===
a2a_example = {
    "protocol": "A2A",
    "direction": "Agent ↔ Agent (양방향 통신)",
    "example": {
        "method": "POST /tasks/send",
        "body": {
            "message": {
                "role": "user",
                "parts": [{"text": "이 문서를 한국어로 번역해주세요"}]
            }
        }
    },
    "use_case": "번역, 코드 리뷰, 데이터 분석 등 에이전트 협업"
}

print("=== MCP 패턴 ===")
print(json.dumps(mcp_example, indent=2, ensure_ascii=False))
print()
print("=== A2A 패턴 ===")
print(json.dumps(a2a_example, indent=2, ensure_ascii=False))
print()
print("핵심 차이: MCP는 '도구 호출', A2A는 '에이전트 간 대화'")
print("완료 ✓")

---
## 섹션 4. A2A Agent Card 정의

**Agent Card**는 에이전트의 명함입니다.  
`GET /.well-known/agent.json` 엔드포인트로 노출되며,  
다른 에이전트가 이를 통해 **능력을 발견**합니다.

In [ ]:
# A2A Agent Card Pydantic 모델 정의

class Skill(BaseModel):
    """에이전트가 수행할 수 있는 개별 능력"""
    id: str = Field(description="스킬 고유 ID")
    name: str = Field(description="스킬 이름")
    description: str = Field(description="스킬 설명")
    tags: list[str] = Field(default_factory=list, description="태그 목록")


class AgentProvider(BaseModel):
    """에이전트를 제공하는 조직 정보"""
    organization: str = Field(description="조직 이름")
    url: str = Field(description="조직 URL")


class AgentCapabilities(BaseModel):
    """에이전트가 지원하는 기능"""
    streaming: bool = Field(default=False, description="스트리밍 응답 지원")
    push_notifications: bool = Field(default=False, description="푸시 알림 지원")
    state_transition_history: bool = Field(default=False, description="상태 전환 이력")


class AgentCard(BaseModel):
    """A2A Agent Card — 에이전트의 명함"""
    name: str = Field(description="에이전트 이름")
    description: str = Field(description="에이전트 설명")
    url: str = Field(description="에이전트 엔드포인트 URL")
    version: str = Field(default="1.0.0", description="에이전트 버전")
    provider: AgentProvider = Field(description="제공자 정보")
    capabilities: AgentCapabilities = Field(
        default_factory=AgentCapabilities, description="지원 기능"
    )
    skills: list[Skill] = Field(default_factory=list, description="스킬 목록")
    default_input_modes: list[str] = Field(
        default=["text/plain"], description="기본 입력 형식"
    )
    default_output_modes: list[str] = Field(
        default=["text/plain"], description="기본 출력 형식"
    )


# Agent Card 생성 예시
translate_card = AgentCard(
    name="번역 에이전트",
    description="한국어 ↔ 영어 전문 번역 에이전트",
    url="http://localhost:8001",
    provider=AgentProvider(
        organization="PDStudio",
        url="https://pdstudio.dev"
    ),
    skills=[
        Skill(
            id="ko-en-translate",
            name="한영 번역",
            description="한국어를 영어로 번역합니다",
            tags=["translation", "korean", "english"]
        ),
        Skill(
            id="en-ko-translate",
            name="영한 번역",
            description="영어를 한국어로 번역합니다",
            tags=["translation", "english", "korean"]
        )
    ]
)

print("=== 번역 에이전트 Agent Card ===")
print(translate_card.model_dump_json(indent=2))
print()
print(f"에이전트: {translate_card.name}")
print(f"스킬 수: {len(translate_card.skills)}개")
for skill in translate_card.skills:
    print(f"  - {skill.name}: {skill.description}")
print("완료 ✓")

---
## 섹션 5. A2A Task / Artifact 모델

**Task**는 A2A에서 에이전트 간 작업의 기본 단위입니다.  
Task는 상태(submitted → working → completed)를 거치며,  
결과물은 **Artifact**로 반환됩니다.

In [ ]:
# A2A Task / Artifact Pydantic 모델 정의

class TaskState(str, Enum):
    """Task 상태 열거형"""
    SUBMITTED = "submitted"    # 작업 제출됨
    WORKING = "working"        # 작업 진행 중
    INPUT_REQUIRED = "input_required"  # 추가 입력 필요
    COMPLETED = "completed"    # 작업 완료
    CANCELED = "canceled"      # 작업 취소됨
    FAILED = "failed"          # 작업 실패


class Part(BaseModel):
    """메시지/아티팩트의 구성 요소"""
    type: str = Field(default="text", description="파트 타입 (text, file 등)")
    text: Optional[str] = Field(default=None, description="텍스트 내용")
    mime_type: Optional[str] = Field(default=None, description="MIME 타입")


class Message(BaseModel):
    """Task 내 메시지"""
    role: str = Field(description="역할 (user 또는 agent)")
    parts: list[Part] = Field(default_factory=list, description="메시지 구성 요소")
    timestamp: str = Field(
        default_factory=lambda: datetime.now().isoformat(),
        description="타임스탬프"
    )


class Artifact(BaseModel):
    """Task의 결과물"""
    name: str = Field(description="아티팩트 이름")
    description: Optional[str] = Field(default=None, description="아티팩트 설명")
    parts: list[Part] = Field(default_factory=list, description="결과물 구성 요소")


class Task(BaseModel):
    """A2A Task — 에이전트 간 작업 단위"""
    id: str = Field(default_factory=lambda: str(uuid.uuid4()), description="Task ID")
    status: TaskState = Field(default=TaskState.SUBMITTED, description="현재 상태")
    history: list[Message] = Field(default_factory=list, description="대화 이력")
    artifacts: list[Artifact] = Field(default_factory=list, description="결과물 목록")
    created_at: str = Field(
        default_factory=lambda: datetime.now().isoformat(),
        description="생성 시각"
    )


# Task 생성 및 상태 전환 시뮬레이션
task = Task()
task.history.append(Message(
    role="user",
    parts=[Part(text="이 문서를 한국어로 번역해주세요")]
))
print(f"=== Task 생성 ===")
print(f"ID: {task.id}")
print(f"상태: {task.status.value}")
print()

# 상태 전환: submitted → working
task.status = TaskState.WORKING
print(f"상태 전환: → {task.status.value}")

# 상태 전환: working → completed (결과물 첨부)
task.status = TaskState.COMPLETED
task.artifacts.append(Artifact(
    name="translation_result",
    description="번역 결과",
    parts=[Part(text="Please translate this document to Korean.")]
))
task.history.append(Message(
    role="agent",
    parts=[Part(text="번역이 완료되었습니다.")]
))
print(f"상태 전환: → {task.status.value}")
print(f"아티팩트 수: {len(task.artifacts)}개")
print(f"대화 이력: {len(task.history)}건")
print()
print("=== Task 전체 구조 ===")
print(task.model_dump_json(indent=2))
print("완료 ✓")

---
## 섹션 6. A2A 서버 구현

A2A 프로토콜을 따르는 HTTP 서버를 구현합니다.  
실제 외부 서비스 없이 **시뮬레이션 에이전트**로 동작합니다.

**엔드포인트**:
- `GET /.well-known/agent.json` — Agent Card 반환
- `POST /tasks/send` — Task 수신 및 처리
- `GET /tasks/{task_id}` — Task 상태 조회

In [ ]:
# A2A 서버 구현 — 요약 에이전트

class A2AServer:
    """A2A 프로토콜을 따르는 에이전트 서버 (시뮬레이션)"""

    def __init__(self, agent_card: AgentCard):
        self.agent_card = agent_card
        self.tasks: dict[str, Task] = {}  # Task 저장소

    def get_agent_card(self) -> dict:
        """Agent Card 반환 (GET /.well-known/agent.json)"""
        return self.agent_card.model_dump()

    def handle_task(self, request_body: dict) -> dict:
        """Task 수신 및 처리 (POST /tasks/send)"""
        # 요청에서 메시지 추출
        msg_data = request_body.get("message", {})
        user_text = ""
        for part in msg_data.get("parts", []):
            if part.get("text"):
                user_text += part["text"]

        # Task 생성
        task = Task()
        task.history.append(Message(
            role="user",
            parts=[Part(text=user_text)]
        ))

        # 에이전트 처리 (시뮬레이션)
        task.status = TaskState.WORKING
        result_text = self._process(user_text)

        # 결과 생성
        task.status = TaskState.COMPLETED
        task.artifacts.append(Artifact(
            name="result",
            description="처리 결과",
            parts=[Part(text=result_text)]
        ))
        task.history.append(Message(
            role="agent",
            parts=[Part(text="작업이 완료되었습니다.")]
        ))

        # 저장
        self.tasks[task.id] = task
        return task.model_dump()

    def get_task(self, task_id: str) -> dict | None:
        """Task 상태 조회 (GET /tasks/{task_id})"""
        task = self.tasks.get(task_id)
        if task:
            return task.model_dump()
        return None

    def _process(self, text: str) -> str:
        """시뮬레이션: 텍스트 요약 처리"""
        # 실제 환경에서는 LLM 호출로 대체
        sentences = text.split(". ")
        if len(sentences) > 3:
            summary = ". ".join(sentences[:3]) + "."
        else:
            summary = text
        return f"[요약] {summary}"


# 요약 에이전트 서버 생성
summarize_card = AgentCard(
    name="요약 에이전트",
    description="텍스트를 핵심 내용으로 요약하는 에이전트",
    url="http://localhost:8001",
    provider=AgentProvider(
        organization="PDStudio",
        url="https://pdstudio.dev"
    ),
    skills=[
        Skill(
            id="summarize",
            name="텍스트 요약",
            description="긴 텍스트를 핵심 내용으로 요약합니다",
            tags=["nlp", "summarization"]
        )
    ]
)

summarize_server = A2AServer(summarize_card)
print(f"A2A 서버 생성: {summarize_card.name}")
print(f"스킬: {[s.name for s in summarize_card.skills]}")
print("완료 ✓")

In [ ]:
# A2A 서버 테스트 — Agent Card 확인 및 Task 전송

# 1. Agent Card 확인
card = summarize_server.get_agent_card()
print("=== Agent Card (GET /.well-known/agent.json) ===")
print(f"이름: {card['name']}")
print(f"설명: {card['description']}")
print(f"URL: {card['url']}")
print()

# 2. Task 전송
test_text = (
    "인공지능은 컴퓨터 과학의 한 분야입니다. "
    "머신러닝은 인공지능의 하위 분야로 데이터에서 학습합니다. "
    "딥러닝은 머신러닝의 한 방법으로 신경망을 사용합니다. "
    "최근 대규모 언어 모델이 주목받고 있습니다. "
    "GPT, Claude, Gemini 등이 대표적인 모델입니다."
)

result = summarize_server.handle_task({
    "message": {
        "role": "user",
        "parts": [{"text": test_text}]
    }
})

print("=== Task 결과 (POST /tasks/send) ===")
print(f"Task ID: {result['id']}")
print(f"상태: {result['status']}")
print(f"결과: {result['artifacts'][0]['parts'][0]['text']}")
print()

# 3. Task 상태 조회
task_status = summarize_server.get_task(result['id'])
print(f"=== Task 조회 (GET /tasks/{result['id'][:8]}...) ===")
print(f"상태: {task_status['status']}")
print(f"이력 수: {len(task_status['history'])}건")
print("완료 ✓")

---
## 섹션 7. A2A 클라이언트 구현

A2A 클라이언트는 **3단계 통신 패턴**을 따릅니다:
1. **Discovery**: Agent Card로 에이전트 능력 확인
2. **Negotiation**: 적합한 에이전트인지 판단
3. **Execution**: Task 전송 → 결과 수신

In [ ]:
# A2A 클라이언트 구현

class A2AClient:
    """A2A 프로토콜 클라이언트"""

    def __init__(self, server: A2AServer = None, base_url: str = None):
        """서버 객체 직접 연결 또는 HTTP URL로 초기화"""
        self.server = server
        self.base_url = base_url
        self.agent_card: dict | None = None

    async def discover(self) -> dict:
        """Phase 1: Agent Card 발견"""
        if self.server:
            # 로컬 시뮬레이션
            self.agent_card = self.server.get_agent_card()
        else:
            # HTTP 호출
            async with httpx.AsyncClient() as client:
                resp = await client.get(
                    f"{self.base_url}/.well-known/agent.json"
                )
                self.agent_card = resp.json()
        return self.agent_card

    def can_handle(self, required_skill: str) -> bool:
        """Phase 2: 에이전트가 특정 스킬을 지원하는지 확인"""
        if not self.agent_card:
            raise ValueError("discover()를 먼저 호출하세요")
        skill_ids = [s["id"] for s in self.agent_card.get("skills", [])]
        return required_skill in skill_ids

    async def send_task(self, text: str) -> dict:
        """Phase 3: Task 전송 및 결과 수신"""
        request_body = {
            "message": {
                "role": "user",
                "parts": [{"text": text}]
            }
        }
        if self.server:
            # 로컬 시뮬레이션
            return self.server.handle_task(request_body)
        else:
            # HTTP 호출
            async with httpx.AsyncClient() as client:
                resp = await client.post(
                    f"{self.base_url}/tasks/send",
                    json=request_body
                )
                return resp.json()

    async def get_task(self, task_id: str) -> dict | None:
        """Task 상태 조회"""
        if self.server:
            return self.server.get_task(task_id)
        else:
            async with httpx.AsyncClient() as client:
                resp = await client.get(
                    f"{self.base_url}/tasks/{task_id}"
                )
                return resp.json()


print("A2AClient 클래스 정의 완료 ✓")

In [ ]:
# A2A 클라이언트 테스트 — 3단계 통신

async def test_a2a_client():
    # 클라이언트 생성 (로컬 서버 연결)
    client = A2AClient(server=summarize_server)

    # Phase 1: Discovery
    print("=== Phase 1: Discovery ===")
    card = await client.discover()
    print(f"에이전트 발견: {card['name']}")
    print(f"설명: {card['description']}")
    print(f"스킬: {[s['name'] for s in card['skills']]}")
    print()

    # Phase 2: Negotiation
    print("=== Phase 2: Negotiation ===")
    can_summarize = client.can_handle("summarize")
    can_translate = client.can_handle("translate")
    print(f"요약 가능: {can_summarize}")
    print(f"번역 가능: {can_translate}")
    print()

    # Phase 3: Execution
    print("=== Phase 3: Execution ===")
    if can_summarize:
        result = await client.send_task(
            "인공지능은 컴퓨터 과학의 한 분야입니다. "
            "머신러닝은 데이터에서 패턴을 학습합니다. "
            "딥러닝은 신경망을 활용한 기법입니다. "
            "트랜스포머 아키텍처가 NLP를 혁신했습니다."
        )
        print(f"Task ID: {result['id'][:8]}...")
        print(f"상태: {result['status']}")
        print(f"결과: {result['artifacts'][0]['parts'][0]['text']}")

        # Task 상태 재조회
        status = await client.get_task(result['id'])
        print(f"재조회 상태: {status['status']}")

    print()
    print("A2A 3단계 통신 완료 ✓")

await test_a2a_client()

---
## 섹션 8. MCP 서버 + A2A 에이전트 결합

**핵심 패턴**: MCP Tool 안에서 A2A 에이전트를 호출합니다.  
이를 통해 MCP 클라이언트(Claude Desktop 등)가 자연스럽게  
A2A 에이전트의 능력을 활용할 수 있습니다.

In [ ]:
# MCP + A2A 결합: MCP Tool에서 A2A 에이전트 호출

from fastmcp import FastMCP

# 번역 에이전트 서버 (A2A)
class TranslateA2AServer(A2AServer):
    """번역 전문 A2A 에이전트 서버"""

    def _process(self, text: str) -> str:
        """시뮬레이션: 한영 번역"""
        # 실제 환경에서는 LLM API 호출로 대체
        translations = {
            "안녕하세요": "Hello",
            "감사합니다": "Thank you",
            "인공지능": "Artificial Intelligence",
            "에이전트": "Agent",
        }
        result = text
        for ko, en in translations.items():
            result = result.replace(ko, en)
        return f"[번역] {result}"


# 번역 에이전트 생성
translate_a2a_server = TranslateA2AServer(translate_card)

# MCP 서버 생성 — A2A 에이전트를 Tool로 감싸기
bridge_mcp = FastMCP("MCP-A2A Bridge Server")

# A2A 클라이언트 인스턴스
translate_client = A2AClient(server=translate_a2a_server)
summarize_client = A2AClient(server=summarize_server)


@bridge_mcp.tool
async def translate_text(text: str) -> str:
    """A2A 번역 에이전트를 통해 텍스트를 번역합니다."""
    # Discovery
    card = await translate_client.discover()
    # Task 전송
    result = await translate_client.send_task(text)
    return result["artifacts"][0]["parts"][0]["text"]


@bridge_mcp.tool
async def summarize_text(text: str) -> str:
    """A2A 요약 에이전트를 통해 텍스트를 요약합니다."""
    card = await summarize_client.discover()
    result = await summarize_client.send_task(text)
    return result["artifacts"][0]["parts"][0]["text"]


print(f"MCP-A2A 브릿지 서버 생성: {bridge_mcp.name}")
print("등록된 MCP Tools:")
print("  - translate_text: A2A 번역 에이전트 호출")
print("  - summarize_text: A2A 요약 에이전트 호출")
print("완료 ✓")

In [ ]:
# MCP + A2A 결합 테스트

async def test_mcp_a2a_bridge():
    # MCP Tool로 A2A 번역 에이전트 호출
    print("=== MCP Tool → A2A 번역 에이전트 ===")
    translate_result = await translate_text("안녕하세요, 인공지능 에이전트입니다")
    print(f"번역 결과: {translate_result}")
    print()

    # MCP Tool로 A2A 요약 에이전트 호출
    print("=== MCP Tool → A2A 요약 에이전트 ===")
    summarize_result = await summarize_text(
        "인공지능은 컴퓨터 과학의 한 분야입니다. "
        "머신러닝은 데이터에서 학습하는 기법입니다. "
        "딥러닝은 신경망 기반의 방법론입니다. "
        "대규모 언어 모델은 텍스트 생성에 뛰어납니다."
    )
    print(f"요약 결과: {summarize_result}")
    print()
    print("MCP + A2A 결합 테스트 완료 ✓")

await test_mcp_a2a_bridge()

---
## 섹션 9. 멀티벤더 에이전트 협업 시나리오

**시나리오**: 오케스트레이터(Claude)가 번역(GPT 시뮬레이션)과  
요약(자체) 에이전트를 A2A로 조율하여 글로벌 고객 지원을 수행합니다.

In [ ]:
# 멀티벤더 협업: GPT 시뮬레이션 에이전트 추가

class GPTSimulatedServer(A2AServer):
    """GPT를 시뮬레이션하는 A2A 에이전트 (코드 리뷰 전문)"""

    def _process(self, text: str) -> str:
        """시뮬레이션: 코드 리뷰 분석"""
        issues = []
        if "eval" in text or "exec" in text:
            issues.append("보안 위험: eval/exec 사용 감지")
        if "password" in text.lower() or "secret" in text.lower():
            issues.append("보안 위험: 하드코딩된 민감 정보 감지")
        if len(text.split("\n")) > 50:
            issues.append("코드 품질: 함수가 너무 깁니다 (50줄 초과)")
        if not issues:
            issues.append("코드 품질 양호: 특별한 이슈 없음")
        return "[GPT 리뷰] " + "; ".join(issues)


# GPT 시뮬레이션 에이전트 생성
gpt_card = AgentCard(
    name="GPT 코드 리뷰 에이전트",
    description="코드 품질과 보안 이슈를 분석하는 에이전트 (GPT 시뮬레이션)",
    url="http://localhost:8002",
    provider=AgentProvider(
        organization="OpenAI (Simulated)",
        url="https://openai.com"
    ),
    skills=[
        Skill(
            id="code-review",
            name="코드 리뷰",
            description="코드 품질과 보안 이슈를 분석합니다",
            tags=["code", "review", "security"]
        )
    ]
)

gpt_server = GPTSimulatedServer(gpt_card)
print(f"GPT 시뮬레이션 에이전트 생성: {gpt_card.name}")
print("완료 ✓")

In [ ]:
# 오케스트레이터: Claude Agent가 멀티 에이전트 협업 조율

class Orchestrator:
    """멀티벤더 A2A 에이전트를 조율하는 오케스트레이터"""

    def __init__(self):
        self.agents: dict[str, A2AClient] = {}

    def register_agent(self, name: str, client: A2AClient):
        """에이전트 등록"""
        self.agents[name] = client

    async def discover_all(self):
        """모든 에이전트 Discovery"""
        print("=== 에이전트 Discovery ===")
        for name, client in self.agents.items():
            card = await client.discover()
            print(f"  [{name}] {card['name']} - 스킬: {[s['name'] for s in card['skills']]}")
        print()

    async def delegate_task(self, agent_name: str, text: str) -> dict:
        """특정 에이전트에게 Task 위임"""
        client = self.agents.get(agent_name)
        if not client:
            raise ValueError(f"에이전트 '{agent_name}'을 찾을 수 없습니다")
        return await client.send_task(text)

    async def run_pipeline(self, text: str) -> dict:
        """파이프라인 실행: 요약 → 번역 → 코드 리뷰 (병렬)"""
        results = {}

        # Step 1: 요약
        if "summarize" in self.agents:
            result = await self.delegate_task("summarize", text)
            results["summarize"] = result["artifacts"][0]["parts"][0]["text"]

        # Step 2: 번역 (요약 결과를)
        if "translate" in self.agents and "summarize" in results:
            result = await self.delegate_task("translate", results["summarize"])
            results["translate"] = result["artifacts"][0]["parts"][0]["text"]

        # Step 3: 코드 리뷰 (원본 텍스트에 코드가 있다면)
        if "code-review" in self.agents:
            result = await self.delegate_task("code-review", text)
            results["code-review"] = result["artifacts"][0]["parts"][0]["text"]

        return results


print("Orchestrator 클래스 정의 완료 ✓")

In [ ]:
# 멀티벤더 협업 테스트

async def test_multi_vendor():
    # 오케스트레이터 생성
    orch = Orchestrator()

    # 에이전트 등록 (각각 다른 벤더)
    orch.register_agent("summarize", A2AClient(server=summarize_server))   # PDStudio
    orch.register_agent("translate", A2AClient(server=translate_a2a_server))  # PDStudio
    orch.register_agent("code-review", A2AClient(server=gpt_server))        # GPT (시뮬레이션)

    # Discovery
    await orch.discover_all()

    # 파이프라인 실행
    test_input = (
        "인공지능은 컴퓨터 과학의 한 분야입니다. "
        "머신러닝은 데이터에서 학습합니다. "
        "딥러닝은 신경망을 활용합니다. "
        "다음 코드에 eval(user_input)이 포함되어 있습니다. "
        "password = 'secret123' 설정이 하드코딩되어 있습니다."
    )

    print("=== 파이프라인 실행 ===")
    print(f"입력: {test_input[:60]}...")
    print()

    results = await orch.run_pipeline(test_input)

    for step, result in results.items():
        print(f"[{step}] {result}")
    print()
    print("멀티벤더 협업 파이프라인 완료 ✓")

await test_multi_vendor()

---
## 섹션 10. Langfuse 통합: A2A 통신 트레이싱

A2A 통신의 전체 흐름을 **Langfuse**로 추적합니다.  
Discovery, Task 전송, 결과 수신 각 단계를 Span으로 기록합니다.

In [ ]:
# Langfuse 통합 A2A 클라이언트

from langfuse import Langfuse
import time

langfuse = Langfuse()


class TracedA2AClient(A2AClient):
    """Langfuse 트레이싱이 통합된 A2A 클라이언트"""

    async def discover_with_trace(self, trace=None) -> dict:
        """트레이싱 포함 Discovery"""
        span = trace.span(name="a2a_discover") if trace else None
        start = time.time()

        card = await self.discover()

        if span:
            span.end(output={
                "agent_name": card["name"],
                "skills": [s["id"] for s in card["skills"]],
                "duration_ms": round((time.time() - start) * 1000)
            })
        return card

    async def send_task_with_trace(self, text: str, trace=None) -> dict:
        """트레이싱 포함 Task 전송"""
        span = trace.span(
            name="a2a_send_task",
            input={"text": text[:200]}
        ) if trace else None
        start = time.time()

        result = await self.send_task(text)

        if span:
            span.end(output={
                "task_id": result["id"],
                "status": result["status"],
                "artifact_count": len(result.get("artifacts", [])),
                "duration_ms": round((time.time() - start) * 1000)
            })
        return result


print("TracedA2AClient 정의 완료 ✓")

In [ ]:
# Langfuse 트레이싱 테스트

async def test_langfuse_tracing():
    # Langfuse 트레이스 시작
    trace = langfuse.trace(
        name="a2a_multi_agent_pipeline",
        metadata={
            "agents": ["summarize", "translate", "code-review"],
            "pipeline": "full"
        }
    )

    # 트레이싱 포함 클라이언트 생성
    traced_summarize = TracedA2AClient(server=summarize_server)
    traced_translate = TracedA2AClient(server=translate_a2a_server)

    # Phase 1: Discovery (트레이싱)
    print("=== Langfuse 트레이싱 테스트 ===")
    card = await traced_summarize.discover_with_trace(trace)
    print(f"Discovery: {card['name']}")

    # Phase 3: Task 전송 (트레이싱)
    result = await traced_summarize.send_task_with_trace(
        "인공지능 기술이 빠르게 발전하고 있습니다. "
        "특히 대규모 언어 모델이 주목받고 있습니다. "
        "멀티 에이전트 시스템이 새로운 패러다임입니다. "
        "A2A 프로토콜이 에이전트 간 통신을 표준화합니다.",
        trace=trace
    )
    print(f"Task 결과: {result['artifacts'][0]['parts'][0]['text']}")

    # 번역 에이전트도 트레이싱
    await traced_translate.discover_with_trace(trace)
    translate_result = await traced_translate.send_task_with_trace(
        "안녕하세요, 인공지능 에이전트입니다",
        trace=trace
    )
    print(f"번역 결과: {translate_result['artifacts'][0]['parts'][0]['text']}")

    # 트레이스 마무리
    trace.update(
        output={
            "summarize_status": result["status"],
            "translate_status": translate_result["status"]
        }
    )

    print()
    print(f"Langfuse trace ID: {trace.id}")
    print("Langfuse 대시보드에서 트레이스를 확인하세요")
    print("Langfuse 트레이싱 완료 ✓")

await test_langfuse_tracing()

---
## 섹션 11. Exercise

### Exercise 1: A2A 에이전트 구축

**목표**: A2A 프로토콜을 따르는 **감성 분석 에이전트**를 구축하세요.

**요구사항**:
1. Agent Card 정의 (이름: "감성 분석 에이전트", 스킬: "sentiment-analysis")
2. `A2AServer`를 상속하여 `_process` 메서드 구현 (긍정/부정/중립 판단)
3. `A2AClient`로 Discovery → Task 전송 테스트
4. 최소 3개의 테스트 입력으로 검증

In [ ]:
# TODO: Exercise 1 구현

# Step 1: Agent Card 정의
sentiment_card = AgentCard(
    name="감성 분석 에이전트",
    description="텍스트의 감성을 분석하는 에이전트",
    url="http://localhost:8003",
    provider=AgentProvider(
        organization="PDStudio",
        url="https://pdstudio.dev"
    ),
    skills=[
        Skill(
            id="sentiment-analysis",
            name="감성 분석",
            description="텍스트의 감성(긍정/부정/중립)을 분석합니다",
            tags=["nlp", "sentiment"]
        )
    ]
)

# Step 2: A2AServer 상속하여 감성 분석 구현
class SentimentA2AServer(A2AServer):
    def _process(self, text: str) -> str:
        # TODO: 감성 분석 로직 구현
        # 힌트: 긍정/부정 키워드 매칭 또는 간단한 규칙 기반
        pass

# Step 3: 서버 생성 및 테스트
# sentiment_server = SentimentA2AServer(sentiment_card)
# client = A2AClient(server=sentiment_server)

# Step 4: 테스트 입력
# test_inputs = [
#     "이 제품 정말 좋아요! 최고입니다!",
#     "서비스가 너무 별로였습니다. 실망이에요.",
#     "보통이었습니다. 특별한 점은 없었어요."
# ]

print("Exercise 1: TODO - 위 주석을 참고하여 구현하세요")

### Exercise 2: MCP + A2A 브릿지 구현

**목표**: Exercise 1의 감성 분석 에이전트를 **MCP Tool로 감싸서** 브릿지를 구현하세요.

**요구사항**:
1. FastMCP 서버에 `analyze_sentiment` Tool 추가
2. Tool 내부에서 A2A 클라이언트로 감성 분석 에이전트 호출
3. Langfuse 트레이싱 추가
4. 테스트 코드로 MCP Tool → A2A 에이전트 호출 검증

In [ ]:
# TODO: Exercise 2 구현

# Step 1: FastMCP 서버 생성
# exercise_mcp = FastMCP("Exercise MCP-A2A Server")

# Step 2: MCP Tool에서 A2A 에이전트 호출
# @exercise_mcp.tool
# async def analyze_sentiment(text: str) -> str:
#     """A2A 감성 분석 에이전트를 통해 텍스트 감성을 분석합니다."""
#     # TODO: A2A 클라이언트로 감성 분석 에이전트 호출
#     pass

# Step 3: Langfuse 트레이싱 추가
# 힌트: TracedA2AClient 패턴 활용

# Step 4: 테스트
# result = await analyze_sentiment("이 영화 정말 감동적이었습니다!")
# print(f"감성 분석 결과: {result}")

print("Exercise 2: TODO - 위 주석을 참고하여 구현하세요")

---
## 마무리

### 오늘 배운 것

- **MCP vs A2A**: 도구 연결(MCP)과 에이전트 간 통신(A2A)의 역할 구분
- **A2A 프로토콜 구조**: Agent Card, Task, Artifact, Message 모델
- **3단계 통신**: Discovery → Negotiation → Execution 패턴
- **A2A 서버/클라이언트**: HTTP 기반 에이전트 간 통신 구현
- **MCP + A2A 결합**: MCP Tool 안에서 A2A 에이전트 호출하는 브릿지 패턴
- **멀티벤더 협업**: Claude + GPT 에이전트가 표준 프로토콜로 협업
- **Langfuse 트레이싱**: A2A 통신 전체 흐름 모니터링

### 다음 에피소드

**EP15. A2A 프로덕션 배포** — A2A 에이전트를 실전 환경에 배포하고 관찰가능성을 확보하는 방법